## Preproccessing of parse data. Merging individual runs into a collected AnnData object

In [1]:
import scanpy as sc
import anndata as ad


import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import os
import glob
import datetime


current_time = f"{datetime.datetime.now():%Y-%m-%d %H:%M}"

/home/frederik/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


I am just following the standard scanpy workflow found [here](https://scanpy.readthedocs.io/en/stable/tutorials/basics/clustering.html) <br>
The raw-data should be located at the pinbot server. 

In [2]:
adata_paths = glob.glob('../h5ad/Parse_single-nuclei/P[0-9]*')
adata_paths

adata_sep = {}

for path in adata_paths:
    name = os.path.basename(path).replace('_anndata.h5ad', '')
    adata_sep[name] = sc.read_h5ad(path)

    adata_sep[name].var_names_make_unique()
    adata_sep[name].obs_names_make_unique()

    # mitochondrial genes, "MT-" for human, "Mt-" for mouse
    adata_sep[name].var["mt"] = adata_sep[name].var_names.str.startswith("MT-")
    # ribosomal genes
    adata_sep[name].var["ribo"] = adata_sep[name].var_names.str.startswith(("RPS", "RPL"))
    # hemoglobin genes
    adata_sep[name].var["hb"] = adata_sep[name].var_names.str.contains("^HB[^(P)]")

    sc.pp.calculate_qc_metrics(adata_sep[name], qc_vars=["mt", "ribo", "hb"],
                                inplace=True, log1p=True)

In [3]:
for adata in adata_sep.values():
    total_counts_p0992 = np.quantile(adata.obs['total_counts'], 0.992)
    sc.pl.violin(adata[adata.obs['total_counts'] < total_counts_p0992],
        ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
        jitter=0.4,
        multi_panel=True)

In [ ]:
adata = sc.concat(adata_sep, label='batch')
# Some donor names has a suffix ('sample_'). This will be removed
adata.obs['sample'] = [name.replace('sample_', '') for name in adata.obs['sample'].astype(str)]
adata.obs['sample'] = adata.obs['sample'].astype('category')

# Gene names has the suffix hg38. This will be removed
adata.var_names = [gene_names.replace('_hg38', '') for gene_names in adata.var_names]

adata.var_names_make_unique()
adata.obs_names_make_unique()

/home/drylab/miniconda3/envs/scrublet/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [ ]:
#adata.write_h5ad('../../h5ad/parse_snRNA-seq_raw.h5ad')